In [ ]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx

from pathlib import Path
from langchain_core.documents import Document #https://reference.langchain.com/python/langchain-core/documents

In [ ]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs") #path para pastas de documentos

documentos = []

for docs in path.iterdir():
    #print (docs.suffix)
    file_dtype = docs.suffix.lower()
    print (file_dtype)

    if file_dtype == ".txt":
        parsing = partition_text(str(docs))

    elif file_dtype == ".pdf":
        parsing = partition_pdf(str(docs))

    elif file_dtype == ".docx":
        parsing = partition_docx(str(docs))

    text = "\n".join([str(el) for el in parsing])

    documentos.append (
        Document (
            page_content = text,
            metadata = {
                "title": docs.name
            }
        )
    )


In [ ]:
documentos

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter (
    chunk_size = 500, #bom ponto de partida
    chunk_overlap = 75 #10 a 20% do chunk size
)

chunking_documentos = text_splitter.split_documents (documentos)

In [ ]:
#chunking_documentos

<h1> Retrieval </h1>

<h2> Sparse Retrieval </h2>

In [ ]:
from langchain_community.retrievers import BM25Retriever

In [ ]:
sparse_retrieval = BM25Retriever.from_documents (chunking_documentos, k = 10)

#display (sparse_retrieval)
#display (sparse_retrieval.docs)

In [ ]:
query = "Qual a relação entre Idyn e Ith?"

sparse = sparse_retrieval.invoke (query)

In [ ]:
#print (len(sparse))
sparse

<h2> Recall@K </h2>

$$
Recall@K = \frac {\text {número de documento recuperados no top K}} {\text {número total de documentos relevantes}} 
$$

In [ ]:
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset.json", "r", encoding = "utf-8") as file:
    dados = json.load (file)

In [ ]:
#x = []

#for q in dados:
    #print (q["query"])
#    x.append(q["doc"])

#print (x)
#print (len(x))

In [ ]:
#teste = sparse_retrieval.invoke (dados[0]["query"])
#for x in teste:
#    print (x.metadata["title"])
#print (f"\n{dados[0]['doc']}")



#for x in teste:
#    if x.metadata["title"] == dados[0]["doc"]:
#        print ("x")
#        break


In [ ]:
### loop de recall@10
### Este Recall funciona especificamente para 1 documento por query

eval_recall = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]
    
    sparse_retr = sparse_retrieval.invoke (query)
    
    hit = 0

    for docs in sparse_retr:
        if docs.metadata["title"] == gold_doc:
            hit = 1
            break
            
    eval_recall.append (hit) 


In [ ]:
eval_recall


In [ ]:
recall_at_10 = sum(eval_recall) / len(eval_recall)

recall_at_10

<h1> Dense Retrieval </h1>

In [ ]:
page_content = []
#Para o modelo Embedding entra apenas texto, não metadata

for i in range (len(chunking_documentos)):
    page_content.append(chunking_documentos[i].page_content)


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
emb_model = r"C:\Users\Admin\Desktop\models\Bi Encoder"

tokenizer = AutoTokenizer.from_pretrained (emb_model)
modelo = AutoModel.from_pretrained (emb_model)

In [ ]:
#Maneira de fazer Embedding
inputs = tokenizer (page_content, return_tensors = "pt", padding = True, truncation = True) #Padding -> Substitui posições de embeddings não utilizados por [PAD] #Truncation -> Corta quando o número de tokens excede o tamanho máximo do modelo

#print (inputs["input_ids"])
#print (len(inputs["input_ids"]))
#print (tokenizer.decode(inputs["input_ids"][0]))
#print (chunking_documentos[0].page_content)

#Modelo pega em cada token e cria uma representação vetorial chamada de Embedding Dimension (Tamanho depende de cada Embedding Model)
with torch.no_grad():
    outputs = modelo (**inputs) 

#print (outputs)
#print (outputs.keys())
#print (outputs.last_hidden_state)
#print (f"Sequence Length do Primeiro Chunk: {outputs.last_hidden_state[0].shape[0]} \nEmbedding Dimension do Primeiro Chunk: {outputs.last_hidden_state[0].shape[1]}")

#Pooling é o processo de juntar vários vetores (neste caso por token) num único vetor fixo que representa todo o texto.
#Isto é importante porque neste momento temos apenas vetores por cada token do respetivo chunk e nós queremos uma representação final do chunk.
attention_mask = inputs["attention_mask"]
mask = attention_mask.unsqueeze (-1) #Não podemos ter em conta os tokens [PAD]
embeddings = (outputs.last_hidden_state * mask).sum(dim = 1) / mask.sum(dim = 1)

#print (embeddings)
#print (f"Texto Chunk 0: {chunking_documentos[0].page_content}\nEmbeddings Chunk 0: {embeddings[0]}")
#print (f"Os {embeddings.shape[0]} Chunks são representados cada, por {embeddings.shape[1]} valores.")

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (chunking_documentos, emb_hf) #Este método calcula os Embeddings e mete logo numa base de dados vetorial

#display (db.index.reconstruct(0))
#display (db.docstore._dict)
#display (db.index.ntotal) #Número de Chunks
#display (db.index_to_docstore_id)
#db.save_local("x")

In [ ]:

dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

retrieval = dense_retrieval.invoke (query) 

In [ ]:
query = "Qual a função do controlador num veículo eléctrico?"

#embededed_query = emb_hf.embed_query (query)

dense = dense_retrieval.invoke (query)

In [ ]:
display (dense)

In [ ]:
eval_recall_2 = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]
    
    dense_retr = dense_retrieval.invoke (query)
    
    hit = 0

    for docs in dense_retr:
        if docs.metadata["title"] == gold_doc:
            hit = 1
            break
            
    eval_recall_2.append (hit) 

In [ ]:
print (sum(eval_recall_2) / len(eval_recall_2))

<h1> Hybrid Retrieval </h1>

In [ ]:
def Reciprocal_Rank_Fusion (rankings, k = 60): #60 é uma variável usada para reduzir o impacto de diferenças pequenas entre ranks
    
    points = {}

    for ranking in rankings:
        #print (ranking)
        for rank, doc in enumerate (ranking):
            
            doc_content = doc.page_content
            id_doc = doc.metadata["title"]
            

            points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)

    
    ranking_final = sorted(points.items(), key=lambda x: x[1], reverse=True)


            #points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)

    return [ #Mudanças para poder englobar o nome dos docs. Importante para calcular Recall@K
        (doc_content, score, doc_id) 
        for (doc_content, score), doc_id in zip(
            ranking_final,
            [doc.metadata["title"] for doc in rankings[0]]
        )
    ]
    # return sorted (points.items(), key = lambda x: x[1], reverse = False) #Dá return dos docs ordenados e respetivas pontuações sorted

In [ ]:
sparse = sparse_retrieval.invoke (query)
dense = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([sparse, dense])

In [ ]:
eval_recall_3 = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]
    
    h_dense_retr = dense_retrieval.invoke (query)
    h_sparse_retr = sparse_retrieval.invoke (query)
    
    print (h_dense_retr)
    print (h_sparse_retr)


    RRF = Reciprocal_Rank_Fusion ([h_dense_retr, h_sparse_retr])
    
    hit = 0

    for docs in RRF:
        
        #print (docs[2])
        if docs[2] == gold_doc:
            hit = 1
            break
            
    eval_recall_3.append (hit) 

In [ ]:
print (sum(eval_recall_3) / len(eval_recall_3))

<h1> MRR </H1>

In [ ]:
rrf_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    rankings = [
        dense_retrieval.invoke(query),
        sparse_retrieval.invoke(query)
    ]

    rrf_ranked = Reciprocal_Rank_Fusion(rankings)

    rr = 0

    for rank, (doc_content, score, doc_id) in enumerate(rrf_ranked, start=1):
        if doc_id == gold_doc:
            rr = 1 / rank
            break

    rrf_scores.append(rr)

mrr_rrf = sum(rrf_scores) / len(rrf_scores)

print("MRR RRF:", mrr_rrf)

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    dense_retr = dense_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Dense:", mrr)

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    sparse_retr = sparse_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Sparse:", mrr)

<h1> Reranking </h1>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder = AutoModelForSequenceClassification.from_pretrained (path)
ce_tokenization = AutoTokenizer.from_pretrained (path)

In [ ]:
#### teste

#features = [query] * len (teste), [t[0] for t in teste]

QUERY = [query] * len (teste)
DOCSS = [t[0] for t in teste]
DOC_NAMES = [t[2] for t in teste]
#print (QUERY)
#print (DOCSS)
#print (DOC_NAMES)

inputs = ce_tokenization (QUERY, DOCSS, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder.eval()
with torch.no_grad():
    logits = cross_encoder (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes
rerank = sorted(
    zip(DOCSS, DOC_NAMES, logits.tolist()),
    key=lambda x: x[2],
    reverse=True
)
display (rerank)

In [ ]:
mrr2_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    rankings = [
        dense_retrieval.invoke(query),
        sparse_retrieval.invoke(query)
    ]

    rrf_ranked = Reciprocal_Rank_Fusion(rankings)


    QUERY = [query] * len (rrf_ranked)
    DOCSS = [t[0] for t in rrf_ranked]
    DOC_NAMES = [t[2] for t in rrf_ranked]
    #print (QUERY)
    #print (DOCSS)
    #print (DOC_NAMES)

    inputs = ce_tokenization (QUERY, DOCSS, return_tensors = "pt", padding = True, truncation = True)
    #print (inputs)

    cross_encoder.eval()
    with torch.no_grad():
        logits = cross_encoder (**inputs).logits
        #print (logits)

    #Organiza os logits com os chunks correspondentes
    rerank = sorted(
        zip(DOCSS, DOC_NAMES, logits.tolist()),
        key=lambda x: x[2],
        reverse=True
    )

    rr = 0

    for rank, doc in enumerate(rerank, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr2_scores.append(rr)

mrr = sum(mrr2_scores) / len(mrr2_scores)

print("MRR Sparse:", mrr)